In [ ]:
%run ./00_Setup_Mount

# =============================================================
# 05_BATCH_INFERENCE.IPYNB (FIXED AUTH VERSION)
# =============================================================



In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import sys

# Agregar src al path
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.ml.scorer import score_dataframe

GOLD_PATH = f"s3a://{BUCKET}/gold/app_inmuebles/"
SCORED_PATH = f"s3a://{BUCKET}/gold/app_inmuebles_scored/"
MODEL_DIR = "/dbfs/mnt/models/champion"

print(f"📖 Cargando Parquet desde: {GOLD_PATH}")

# Forzar credenciales S3 en el Spark session para prevenir 403 anónimos
for k, v in S3_OPTIONS.items():
    spark.conf.set(k, v)

# Cargar con opciones explícitas
df_gold_spark = spark.read.options(**S3_OPTIONS).parquet(GOLD_PATH)

print("⬇️ Bajando a Pandas...")
# La acción toPandas() ahora detectará las credenciales del spark.conf
df_pandas = df_gold_spark.toPandas()
print(f"✅ {len(df_pandas)} registros cargados.")


In [ ]:
import glob

print(f"🔍 Buscando bundle en {MODEL_DIR}...")
pickles = glob.glob(os.path.join(MODEL_DIR, "bundle_v*.pkl"))
if not pickles:
    pickles = glob.glob("models/bundle_v*.pkl")

if not pickles:
    raise FileNotFoundError("No se encontró bundle de modelo.")

latest_bundle_path = max(pickles, key=os.path.getmtime)
print(f"📦 Cargando: {os.path.basename(latest_bundle_path)}")

with open(latest_bundle_path, "rb") as f:
    bundle = pickle.load(f)
    
print(f"✅ Modelo {bundle.get('model_version', 'v?')} cargado.")


In [ ]:
print("⚙️ Generando rentabilidad...")
df_scored_pandas = score_dataframe(df_pandas, bundle)
print("✅ Completado.")
display(df_scored_pandas[["titulo", "city_token", "precio_num", "precio_predicho", "rentabilidad_potencial", "estado_inversion"]].head(10))


In [ ]:
print("⬆️ Guardando resultado...")

# Limpieza básica de tipos
for col in df_scored_pandas.columns:
    if df_scored_pandas[col].dtype == "object":
        df_scored_pandas[col] = df_scored_pandas[col].fillna("").astype(str)

df_scored_spark = spark.createDataFrame(df_scored_pandas)

# Guardar como Delta (formato nativo Databricks para la salida)
print(f"💾 Guardando en: {SCORED_PATH}")
writer = df_scored_spark.write.format("delta").mode("overwrite").options(**S3_OPTIONS).option("overwriteSchema", "true")
writer.save(SCORED_PATH)

print("🎉 Misión Inferencia Batch Finalizada Exitosamente.")
